# ShopSense: E-Commerce Customer Intelligence & Growth Platform
**Author:** Uma  
**Domain:** E-Commerce Analytics, Customer Intelligence, Machine Learning  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Public real-world transaction logs)  
**Tools & Stack:** Python (Pandas, Scikit-Learn), Microsoft SQL Server, FastAPI, JavaScript, Power BI  

---

## 1. Project Overview & Business Problem
In high-volume e-commerce, businesses accumulate vast transactional logs (orders, payments, reviews, product catalogs) but struggle to translate raw records into proactive growth decisions. 

**ShopSense** is an enterprise customer intelligence platform designed to:
1. **Identify Customer Churn & Inactivity:** Detect which buyers are drifting away before they stop purchasing permanently.
2. **Segment Customer Value (RFM & Clustering):** Differentiate one-time bargain hunters from high-value VIP champions.
3. **Recommend Products:** Suggest relevant cross-sell items based on purchase history and item similarity.
4. **Deliver Executive KPIs:** Provide real-time operational health metrics (delivery lag, revenue trends, review ratings).

---

## 2. Technical Architecture Pipeline
[Raw Olist CSV Ingestion (9 Datasets)]
↓
[Data Validation & Quality Auditing (Python / Jupyter)]
↓
[Relational Database Staging & Normalization (Microsoft SQL Server)]
↓
[Analytical Views, Window Functions & CTEs (SQL Server)]
↓
[Data Science Layer: Feature Engineering, RFM, Scikit-Learn ML Models]
↓
[API Service Layer: FastAPI REST Endpoints]
↓
[Client Presentation: Interactive Frontend UI (JavaScript) & Power BI Dashboards]

---

## 3. Environment & Workspace Specifications
- **Operating System:** Windows 11
- **Hardware Profile:** 8 GB RAM (memory-optimized pipeline)
- **Database Engine:** Microsoft SQL Server Developer/Express with SSMS
- **Data Science Lab:** Jupyter Notebooks & Python Virtual Environment (`venv`)
- **Version Control:** Git configured with `.gitignore` to prevent credential and raw dataset leakage.

---

## 4. Raw Data Audit & Profile Summary
An automated audit of all 9 transactional tables confirmed row counts, schemas, and null distributions:

| Dataset Name | Total Rows | Total Columns | Null Values | Primary Role in ShopSense |
| :--- | :--- | :--- | :--- | :--- |
| `olist_customers_dataset.csv` | 99,441 | 5 | 0 | Customer identity mapping & geography |
| `olist_orders_dataset.csv` | 99,441 | 8 | 4,908 | Transaction timestamps & order statuses |
| `olist_order_items_dataset.csv` | 112,650 | 7 | 0 | Basket item details, prices, and freight |
| `olist_order_payments_dataset.csv` | 103,886 | 5 | 0 | Payment methods, installments, transaction values |
| `olist_order_reviews_dataset.csv` | 99,224 | 7 | 145,903 | 1–5 star customer ratings & survey feedback |
| `olist_products_dataset.csv` | 32,951 | 9 | 2,448 | Product dimensions, categories, and attributes |
| `olist_sellers_dataset.csv` | 3,095 | 4 | 0 | Merchant IDs and regional locations |
| `olist_geolocation_dataset.csv` | 1,000,163 | 5 | 0 | Zip code prefix mapping to coordinates |
| `product_category_name_translation.csv` | 71 | 2 | 0 | Portuguese to English category dictionary |

---

## 5. Core Transactional Discoveries & Business Rules

### A. The Customer Identity Gotcha (`customer_id` vs `customer_unique_id`)
- **`customer_id` (Order-level token):** Unique per purchase transaction (99,441 records).
- **`customer_unique_id` (Entity key):** Represents the actual human consumer (96,096 distinct individuals).
- **Strategic Impact:** Repeat purchase analysis, churn modeling, and RFM calculations must strictly group by `customer_unique_id`. Aggregating by `customer_id` produces a false repeat rate of 0%.

### B. Order Fulfillment Rule
- Over **97%** of transactions have `order_status = 'delivered'`.
- Unfulfilled orders (`canceled`, `unavailable`, `invoiced`) must be excluded when computing gross revenue, customer spending, and lifetime value.

### C. Operational Time Horizon
- **Date Range:** September 4, 2016 – October 17, 2018 (~25 months / 773 days).
- **Baseline Date for RFM:** Recency calculations must anchor to the maximum date recorded in the dataset (`2018-10-17`), not the present calendar day.

## 6. Financial, Product & Sentiment Breakdown

### A. Payment Distribution
- **Credit Card Dominance:** Accounts for >75% of transaction volume, followed by Boleto vouchers (~19%).
- **Installment Behavior:** Brazilian consumers frequently leverage installment financing (ranging from 1 to 24 installments), an important feature for basket size and credit-driven purchasing.

### B. Product Catalog & Translation
- Original catalog classifications stored in Portuguese (`cama_mesa_banho`, `beleza_saude`).
- Normalized against `product_category_name_translation` to establish English taxonomies (`bed_bath_table`, `health_beauty`, `sports_leisure`).
- Top categories align with everyday lifestyle and home furnishings.

### C. Customer Feedback Sentiment
- **Mean Score:** ~4.08 / 5.0.
- **Polarity:** 57% 5-star reviews (high satisfaction) vs. 14% 1-star reviews (severe friction).
- **Predictive Role:** Review sentiment acts as a leading indicator of churn risk and fulfillment dissatisfaction.

## 7. Relational Database Architecture (Microsoft SQL Server)

### A. Database Overview
- **Database Engine:** Microsoft SQL Server Developer / Express Edition
- **Database Name:** `ShopSenseDB`
- **Objective:** Establish an enterprise-grade relational structure ensuring referential integrity, indexing efficiency, and ACID compliance for analytical querying.

### B. Relational Schema Design (Entity-Relationship Blueprint)
The database structure establishes normalized relational integrity anchored by the central `orders` entity:
- **`customers`** (PK: `customer_id`) ➔ 1:N ➔ **`orders`** (FK: `customer_id`)
- **`orders`** (PK: `order_id`) ➔ 1:N ➔ **`order_items`** (FK: `order_id`)
- **`orders`** (PK: `order_id`) ➔ 1:N ➔ **`order_payments`** (FK: `order_id`)
- **`orders`** (PK: `order_id`) ➔ 1:N ➔ **`order_reviews`** (FK: `order_id`)
- **`products`** (PK: `product_id`) ➔ 1:N ➔ **`order_items`** (FK: `product_id`)
- **`sellers`** (PK: `seller_id`) ➔ 1:N ➔ **`order_items`** (FK: `seller_id`)
- **`product_category_name_translation`** ➔ 1:N ➔ **`products`**

### C. Integrity & Performance Rules
1. **Primary Key Constraints:** Enforce strict record uniqueness on entity tables.
2. **Foreign Key Constraints:** Prevent orphaned transaction records.
3. **Data Type Optimization:** Explicit storage allocation using appropriate types (`DATETIME2`, `DECIMAL(10,2)`, `VARCHAR(50)`) instead of unconstrained string representations.

## 8. Relational Data Definition (DDL) & Schema Specifications

### A. Implemented Relational Tables
We executed standard DDL scripts creating 9 core relational tables in `ShopSenseDB`:
1. `dbo.customers`: PK (`customer_id`). Maps individual customer order tokens to unique human IDs.
2. `dbo.orders`: PK (`order_id`). Central transactional ledger with precise `DATETIME2` fulfillment milestones.
3. `dbo.order_items`: Composite PK (`order_id`, `order_item_id`). Tracks line-item prices, seller mappings, and shipping limits.
4. `dbo.order_payments`: Order transaction payment types, installment splits, and `DECIMAL(10,2)` values.
5. `dbo.order_reviews`: Survey ratings (1–5 scale) and Unicode text feedback (`NVARCHAR(MAX)`).
6. `dbo.products`: PK (`product_id`). Dimensions, weights, and category taxonomies.
7. `dbo.sellers`: PK (`seller_id`). Merchant operational locations.
8. `dbo.product_category_translation`: PK (`product_category_name`). Cross-reference dictionary for Portuguese-to-English translations.
9. `dbo.geolocation`: Spatial location lookup mapping postal prefixes to geographic coordinates.

### B. Engineering & Storage Decisions
- **Financial Precision:** Currency fields (`price`, `freight_value`, `payment_value`) use fixed-point `DECIMAL(10,2)` to prevent floating-point calculation drift.
- **Multilingual Support:** Textual feedback fields use `NVARCHAR` to properly preserve Brazilian Portuguese diacritics and special characters (`ã`, `ç`, `é`).
- **Composite Key Identification:** `order_items` enforces a composite primary key (`order_id` + `order_item_id`), representing individual items within a single basket.

## 10. Analytical SQL Views & Executive Business KPIs

### A. Master Analytical View (`dbo.vw_order_details`)
To eliminate repetitive multi-table joins, we constructed a centralized analytical view joining 6 core tables:
- **Relational Joins:** `orders` ➔ `customers` ➔ `order_items` ➔ `products` ➔ `product_category_translation`.
- **Derived Metrics:**
  - `delivery_days`: Elapsed calendar days between purchase and customer receipt using SQL `DATEDIFF`.
  - `is_delayed`: Binary indicator identifying orders received past the carrier's estimated delivery date.
  - `total_item_value`: Item gross financial footprint (`price` + `freight_value`).

### B. Core E-Commerce Business KPIs (Delivered Cohort)
- **Gross Market Volume (GMV):** R$ 15.42 Million
- **Product Revenue:** R$ 13.22 Million (Freight accounts for ~R$ 2.20M in logistics cashflow).
- **Delivered Transaction Volume:** 96,478 orders.
- **Unique Customer Base:** 93,358 distinct purchasers.
- **Average Order Value (AOV):** R$ 159.85.
- **Fulfillment Velocity:** 12.5 days average delivery time across Brazil.
- **Logistics Defect Rate:** 6.7% of fulfilled orders arrived past the promised delivery SLA.

## 11. Advanced SQL Analytics: Window Functions & Longitudinal Trends

### A. Month-over-Month (MoM) Growth Analysis
- **Technique:** Leveraged CTEs combined with the `LAG()` analytical window function to track monthly trajectory without redundant self-joins.
- **Trend Discovery:** Platform experienced hyper-growth in 2017 (surpassing R$ 700k/month by Q4 2017) driven by the rapid onboarding of merchants, stabilizing into steady operations across 2018 (averaging ~R$ 900k–R$ 1.0M/month).

### B. Regional Category Dominance (`DENSE_RANK() OVER PARTITION`)
- Partitioned sales rankings across the top 3 macroeconomic hubs (São Paulo `SP`, Rio de Janeiro `RJ`, Minas Gerais `MG`):
  - `SP`: Dominated by `bed_bath_table`, `health_beauty`, and `sports_leisure`.
  - `RJ`: High demand for `watches_gifts` and `computers_accessories`.
  - Proves regional merchandise preferences vary across state boundaries.

### C. Repeat Purchase Retention Reality
- **One-off Consumers:** 96.9% of purchasers completed exactly one order.
- **Repeat Cohort:** ~3.1% (~2,900 customers) completed 2 or more transactions.
- **Strategic Imperative:** The platform relies heavily on continuous paid acquisition. Developing automated churn risk detection and personalized cross-selling is essential for increasing retention.

## 12. Customer Intelligence & RFM Segmentation

### A. Methodology & The Frequency Binning Nuance
- **Recency ($R$):** Days elapsed from customer's latest purchase to analysis anchor date (`2018-10-18`). Quintile binned (Score 1 to 5).
- **Frequency ($F$):** Total delivered orders. Due to severe positive skew (~97% single purchases), standard quantile binning fails (`ValueError: Bin edges must be unique`). We engineered a defensible discrete mapping:
  - 1 Order ➔ Score 1 (Single-order buyer)
  - 2 Orders ➔ Score 3 (Returning repeat buyer)
  - $\ge$ 3 Orders ➔ Score 5 (High-frequency buyer)
- **Monetary ($M$):** Gross customer spend (merchandise + freight). Quintile binned (Score 1 to 5).

### B. Segment Definitions & Strategic Action Plan
1. **Champions (VIP):** High R, High F, High M. Top priority for loyalty rewards and VIP early access.
2. **Loyal Customers:** Frequent repeat buyers with consistent spend. Candidates for cross-selling and subscription programs.
3. **Promising New:** High R, purchased recently for the first time. Target with onboarding nurture sequences to trigger order #2.
4. **At Risk / High Value Lost:** Low R (inactive for >250 days) but historically high spend. Immediate priority for win-back discounts.
5. **Lost / Hibernating:** Low R, single purchase, low spend. Suppress aggressive ad spend to prevent wasted marketing capital.

### C. Relational Persistence
- Segment classifications and RFM scores exported back into SQL Server as `dbo.customer_rfm_segments` (93,358 customer records).

## 13. Machine Learning: Churn Target Formulation & Temporal Feature Engineering

### A. Non-Contractual Churn Problem Definition
- In retail e-commerce, customer churn is unobserved (no contractual cancellation event).
- We formulated a defensible **Temporal Observation Window** to model churn without data leakage:
  - **Cutoff Date ($T$):** `2018-04-01`
  - **Feature Observation Window:** All orders placed prior to $T$.
  - **Performance Evaluation Horizon:** 6-Month forward window (`2018-04-01` to `2018-10-17`).
  - **Target Definition ($Y$):**
    - $Y = 0$ (Retained / Active): Customer repurchased during the 6-month prediction horizon.
    - $Y = 1$ (Inactive / Churned): Customer placed zero follow-up transactions.

### B. Engineered Features (Zero Leakage)
1. `recency_days`: Days elapsed between customer's most recent order and the cutoff timestamp.
2. `historical_orders`: Prior distinct order count.
3. `total_spend`: Cumulative historical gross spend.
4. `avg_order_value`: Average spend per historical basket.
5. `avg_review_score`: Mean feedback rating given by customer.
6. `avg_delivery_days`: Fulfillment lag experienced by customer.
7. `delayed_orders_count`: Carrier delivery SLA breach count.

### C. Class Imbalance Consideration
- Retained Buyers ($Y=0$): ~2.0%
- Churned Buyers ($Y=1$): ~98.0%
- **Evaluation Strategy:** Standard classification accuracy is misleading. Models must be tuned and evaluated prioritizing **ROC-AUC, Precision-Recall AUC (PR-AUC), and Recall** to catch at-risk buyers.

## 14. Machine Learning: Model Training, Evaluation & Explainability

### A. Experimental Setup & Preprocessing
- **Partitioning:** Stratified 80/20 train-test split (`stratify=y`) preserving the 99.03% churn / 0.97% retained distribution across 62,299 historical customer accounts.
- **Data Imputation:** Imputed fulfillment latency nulls using median training metrics to guarantee zero leakage.
- **Class Balancing:** Configured balanced cost-sensitive weights (`class_weight='balanced'`) to penalize minority misclassifications.

### B. Authentic Model Benchmarking Results
Three algorithms were evaluated on holdout test data prioritizing rank discrimination:

| Candidate Algorithm | ROC-AUC Score | PR-AUC Score | Operational Assessment |
| :--- | :--- | :--- | :--- |
| **Logistic Regression (Balanced)** | **0.5921** | **0.9931** | **Best discriminator under extreme class imbalance** |
| Hist Gradient Boosting | 0.5810 | 0.9924 | Captures non-linear thresholds |
| Random Forest (Balanced) | 0.5458 | 0.9917 | Prone to overfitting on extreme skew |

### C. Explainable Machine Learning (Feature Signals)
- **`recency_days`:** Primary positive coefficient driving churn risk. Extended dormancy strongly indicates buyer disengagement.
- **`historical_orders` & `total_spend`:** Negative coefficients acting as buffers; higher historical activity correlates with retention.

### D. Model Serialization (Production Readiness)
- Production model (`churn_model.pkl`), feature scaler (`churn_scaler.pkl`), and schema column order exported into `models/` for live API inference.

## 15. Product Intelligence & Hybrid Recommendation Engine

### A. The E-Commerce Matrix Sparsity Challenge
- Over 96% of retail customers completed exactly one transaction, rendering traditional User-to-User Collaborative Filtering ineffective due to sparse user-item interaction matrices.

### B. Hybrid Recommendation Architecture
We engineered a production-viable dual-tier recommender:
1. **Market Basket Co-Occurrence:** Self-joined multi-item transaction baskets (`order_items`) to discover product pairs frequently co-purchased.
2. **Category Best-Seller Fallback (Cold-Start Resolution):** For single-purchase customers with unobserved item pairs, the engine queries the highest-rated ($\ge 4.0\star$) and highest-volume items within the buyer’s historic category.

### C. Explainability & Transparent Match Scoring
- Recommendations return clearly defined affinity scores (75%–95%) derived from observed co-purchase frequencies and category popularity.
- Every recommendation produces an interpretable business justification (e.g., *"Frequently bought together with your previous order"* or *"Top-rated best-seller in your favorite category"*).

### D. Production Serialization
- Precomputed co-occurrence graphs and category popularity lookup tables serialized into `models/recommendation_engine.pkl` for low-latency (<50ms) API serving.